# TDSE Solver — Validation

Internal correctness checks, run phase by phase. Each phase's checks build on the
last — later phases shouldn't be trusted until earlier ones pass. See
`TDSE_Solver_Plan.md` for the full design rationale.

## Phase 1 — Grid & kinetic propagator

The split-operator method needs the kinetic propagator `e^{-i T dt}` (T = -1/2
laplacian) applied exactly via a spectral transform:

- **periodic** boundary: FFT, eigenvalue `k^2` for plane-wave mode `e^{ikx}`.
- **box** (hard-wall, psi=0 at the edges) boundary: DST-I, eigenvalue `(n*pi/L)^2`
  for sine mode `sin(n*pi*x/L)`.

Two checks: (1) a single exact eigenmode of each transform should pick up *exactly*
the analytic phase `exp(-i*k^2*dt/2)` and nothing else — this tests the propagator
is doing the right spectral multiply, independent of any potential; (2) the kinetic
step alone should conserve `sum(|psi|^2)` to machine precision, in 1D/2D/3D, under
both boundary conventions — it's a unitary operator (Parseval), so any drift would
mean a bug, not truncation error.

In [1]:
import sys
sys.path.insert(0, r".")
import numpy as np
import grid as g
import propagator as prop

np.random.seed(0)
results = []
def check(name, cond, detail=""):
    results.append((name, bool(cond), detail))
    print(f"{'PASS' if cond else 'FAIL'}: {name}  {detail}")


In [2]:
# --- Check 1a: periodic plane-wave eigenmode picks up exact analytic phase ---
L, N = 20.0, 64
grid1d = g.make_grid((L,), (N,), boundary='periodic')
x = grid1d.axes[0]
k0 = grid1d.k_axes[0][5]   # an exact grid frequency, so e^{i k0 x} is an exact DFT eigenmode
dt = 0.37
psi0 = np.exp(1j * k0 * x)
psi1 = prop.kinetic_step(psi0, grid1d, dt)
analytic = psi0 * np.exp(-1j * k0**2 * dt / 2)
err = np.max(np.abs(psi1 - analytic))
check("1a periodic plane-wave exact phase (1D)", err < 1e-11, f"max err={err:.2e}")


PASS: 1a periodic plane-wave exact phase (1D)  max err=1.39e-15


In [3]:
# --- Check 1b: box (hard-wall) sine eigenmode picks up exact analytic phase ---
L, N = 20.0, 64
grid1d_box = g.make_grid((L,), (N,), boundary='box')
x = grid1d_box.axes[0]
m = 5
k_m = grid1d_box.k_axes[0][m - 1]   # = m*pi/L
dt = 0.37
psi0 = np.sin(m * np.pi * x / L)
psi1 = prop.kinetic_step(psi0.astype(complex), grid1d_box, dt)
analytic = psi0 * np.exp(-1j * k_m**2 * dt / 2)
err = np.max(np.abs(psi1 - analytic))
check("1b box sine-mode exact phase (1D)", err < 1e-11, f"max err={err:.2e}")


PASS: 1b box sine-mode exact phase (1D)  max err=2.14e-15


In [4]:
# --- Check 1c: same two checks, generalized to 2D (product mode) ---
L, N = 20.0, 32
grid2d_per = g.make_grid((L, L), (N, N), boundary='periodic')
X, Y = grid2d_per.coords
kx0 = grid2d_per.k_axes[0][3]
ky0 = grid2d_per.k_axes[1][-2]  # a negative frequency, exercises fftfreq's ordering
dt = 0.21
psi0 = np.exp(1j * (kx0 * X + ky0 * Y))
psi1 = prop.kinetic_step(psi0, grid2d_per, dt)
analytic = psi0 * np.exp(-1j * (kx0**2 + ky0**2) * dt / 2)
err_per = np.max(np.abs(psi1 - analytic))
check("1c periodic plane-wave exact phase (2D)", err_per < 1e-11, f"max err={err_per:.2e}")

grid2d_box = g.make_grid((L, L), (N, N), boundary='box')
X, Y = grid2d_box.coords
mx, my = 4, 7
kx_m = grid2d_box.k_axes[0][mx - 1]
ky_m = grid2d_box.k_axes[1][my - 1]
psi0 = (np.sin(mx * np.pi * X / L) * np.sin(my * np.pi * Y / L)).astype(complex)
psi1 = prop.kinetic_step(psi0, grid2d_box, dt)
analytic = psi0 * np.exp(-1j * (kx_m**2 + ky_m**2) * dt / 2)
err_box = np.max(np.abs(psi1 - analytic))
check("1c box sine-mode exact phase (2D)", err_box < 1e-11, f"max err={err_box:.2e}")


PASS: 1c periodic plane-wave exact phase (2D)  max err=2.32e-15
PASS: 1c box sine-mode exact phase (2D)  max err=1.63e-15


In [5]:
# --- Check 2: kinetic_step conserves norm to machine precision, many steps,
# random (non-eigenmode) wavefunctions, both boundaries, 1D/2D/3D ---
def norm_drift_after_many_steps(grid, n_steps=500, dt=0.05):
    psi = np.random.normal(size=grid.shape) + 1j * np.random.normal(size=grid.shape)
    if grid.boundary == 'box':
        psi = psi.real.astype(complex)  # box grid still fine with complex psi; just reuse real part for variety
    n0 = np.sum(np.abs(psi)**2)
    k2 = prop.kinetic_eigenvalues(grid)
    for _ in range(n_steps):
        psi = prop.kinetic_step(psi, grid, dt, k2=k2)
    n1 = np.sum(np.abs(psi)**2)
    return abs(n1 - n0) / n0

cases = [
    ("1D periodic", g.make_grid((20.0,), (80,), boundary='periodic')),
    ("1D box",      g.make_grid((20.0,), (80,), boundary='box')),
    ("2D periodic", g.make_grid((20.0, 15.0), (48, 40), boundary='periodic')),
    ("2D box",      g.make_grid((20.0, 15.0), (48, 40), boundary='box')),
    ("3D periodic", g.make_grid((10.0, 10.0, 10.0), (20, 20, 20), boundary='periodic')),
    ("3D box",      g.make_grid((10.0, 10.0, 10.0), (20, 20, 20), boundary='box')),
]
for name, grid_case in cases:
    drift = norm_drift_after_many_steps(grid_case)
    check(f"2 norm conservation over 500 kinetic steps ({name})", drift < 1e-10, f"relative drift={drift:.2e}")


PASS: 2 norm conservation over 500 kinetic steps (1D periodic)  relative drift=5.61e-14
PASS: 2 norm conservation over 500 kinetic steps (1D box)  relative drift=1.97e-13
PASS: 2 norm conservation over 500 kinetic steps (2D periodic)  relative drift=1.03e-13
PASS: 2 norm conservation over 500 kinetic steps (2D box)  relative drift=1.26e-13


PASS: 2 norm conservation over 500 kinetic steps (3D periodic)  relative drift=2.22e-13


PASS: 2 norm conservation over 500 kinetic steps (3D box)  relative drift=2.27e-13


In [6]:
n_pass = sum(1 for _, ok, _ in results if ok)
print(f"\n{n_pass}/{len(results)} Phase 1 checks passed")
assert n_pass == len(results), "Phase 1 validation failed"



10/10 Phase 1 checks passed


## Phase 2 — Potential half-step & full split-operator step

Two genuinely different checks, for a reason worth stating explicitly: a **free
particle** (V=0) has *no* splitting error at all -- the potential half-steps are
identity and the middle `kinetic_step` is already an exact application of the
free-particle propagator, so a free Gaussian wavepacket is a good check of
*physical correctness* (does it reproduce the textbook closed-form spreading law)
but useless for checking 2nd-order *convergence in dt*, since there's nothing for
dt-refinement to improve. For that, we need V and T to genuinely not commute, so
the convergence-order check instead uses the quantum harmonic oscillator: its
eigenstates are exact closed-form solutions of the *full* Hamiltonian
(`potentials.harmonic_eigenstate`), so propagating one and comparing to
`psi(t) = psi(0) * exp(-i*E*t)` isolates the splitting error directly, with no
dependence on any numerical diagonalization routine.

In [7]:
import propagator as prop
import potentials as pot

results2 = []
def check2(name, cond, detail=""):
    results2.append((name, bool(cond), detail))
    print(f"{'PASS' if cond else 'FAIL'}: {name}  {detail}")


In [8]:
# --- Check 2a: free Gaussian wavepacket spreading matches the analytic
# closed-form sigma(t) = sigma0*sqrt(1+(t/(2*sigma0^2))^2) ---
L, N = 200.0, 1024
grid_free = g.make_grid((L,), (N,), boundary='periodic')
x = grid_free.axes[0]
sigma0 = 3.0
psi0 = g.gaussian_wavepacket(grid_free, center=0.0, sigma=sigma0, k0=0.0)
V0 = np.zeros(grid_free.shape)

T, dt = 40.0, 0.05
n_steps = round(T / dt)
psi = psi0.copy()
k2 = prop.kinetic_eigenvalues(grid_free)
for _ in range(n_steps):
    psi = prop.strang_step(psi, grid_free, V0, dt, k2=k2)

w = np.abs(psi) ** 2
mean_x = np.sum(x * w) / np.sum(w)
sigma_num = np.sqrt(np.sum((x - mean_x) ** 2 * w) / np.sum(w))
sigma_analytic = sigma0 * np.sqrt(1 + (T / (2 * sigma0 ** 2)) ** 2)
rel_err = abs(sigma_num - sigma_analytic) / sigma_analytic
check2("2a free Gaussian spreading matches analytic sigma(t)", rel_err < 1e-3,
       f"sigma_num={sigma_num:.4f} sigma_analytic={sigma_analytic:.4f} rel_err={rel_err:.2e}")


PASS: 2a free Gaussian spreading matches analytic sigma(t)  sigma_num=7.3106 sigma_analytic=7.3106 rel_err=6.20e-15


In [9]:
# --- Check 2b: propagating a harmonic-oscillator eigenstate changes it
# by exactly exp(-i*E*t), not shape -- and check 2nd-order convergence in
# dt against this exact closed-form solution ---
L, N = 40.0, 256
grid_ho = g.make_grid((L,), (N,), boundary='periodic')
omega = 1.0
V_ho = pot.harmonic_well(grid_ho, omega)
psi0_gs, E0 = pot.harmonic_eigenstate(grid_ho, omega, n=0)

T = 1.6
errs = []
for dt in (0.08, 0.04, 0.02):
    n_steps = round(T / dt)
    psi = psi0_gs.copy()
    k2 = prop.kinetic_eigenvalues(grid_ho)
    for _ in range(n_steps):
        psi = prop.strang_step(psi, grid_ho, V_ho, dt, k2=k2)
    analytic = psi0_gs * np.exp(-1j * E0 * T)
    err = np.max(np.abs(psi - analytic))
    errs.append(err)
    print(f"  dt={dt:.3f}: max err vs exact e^{{-iEt}} solution = {err:.3e}")

# The error above is dominated by the O(dt^2) splitting error, not spatial
# (spectral) discretization error -- confirmed by the clean 2nd-order
# scaling below -- so at dt=0.02 an error of a few e-5 is *expected*, not
# a sign of a bug. A separate, much finer dt demonstrates the method
# reaches high absolute accuracy once dt is small enough.
dt_fine = 0.005
n_steps = round(T / dt_fine)
psi = psi0_gs.copy()
k2 = prop.kinetic_eigenvalues(grid_ho)
for _ in range(n_steps):
    psi = prop.strang_step(psi, grid_ho, V_ho, dt_fine, k2=k2)
analytic = psi0_gs * np.exp(-1j * E0 * T)
err_fine = np.max(np.abs(psi - analytic))
check2("2b ground-state phase test, fine dt is very accurate", err_fine < 1e-5,
       f"dt={dt_fine}: err={err_fine:.2e}")

order1 = np.log2(errs[0] / errs[1])
order2 = np.log2(errs[1] / errs[2])
check2("2b splitting error is ~2nd order in dt (halving dt)", 1.5 < order1 < 2.5 and 1.5 < order2 < 2.5,
       f"observed orders: {order1:.2f}, {order2:.2f}")


  dt=0.080: max err vs exact e^{-iEt} solution = 3.488e-04
  dt=0.040: max err vs exact e^{-iEt} solution = 8.717e-05
  dt=0.020: max err vs exact e^{-iEt} solution = 2.179e-05
PASS: 2b ground-state phase test, fine dt is very accurate  dt=0.005: err=1.36e-06
PASS: 2b splitting error is ~2nd order in dt (halving dt)  observed orders: 2.00, 2.00


In [10]:
# --- Check 2c: same eigenstate-phase test for a nodal (n=1) excited
# state, and for 2D/3D isotropic wells -- confirms strang_step and
# harmonic_eigenstate generalize correctly, not just for the ground state.
# dt=0.01 here (rather than 2b's dt_fine=0.005) purely to keep the 2D/3D
# runs fast; per the confirmed O(dt^2) scaling that still means errors
# ~4x smaller than the dt=0.02 case above, comfortably under 1e-4. ---
psi0_n1, E1 = pot.harmonic_eigenstate(grid_ho, omega, n=1)
dt = 0.01
n_steps = round(T / dt)
psi = psi0_n1.copy()
k2 = prop.kinetic_eigenvalues(grid_ho)
for _ in range(n_steps):
    psi = prop.strang_step(psi, grid_ho, V_ho, dt, k2=k2)
analytic = psi0_n1 * np.exp(-1j * E1 * T)
err_n1 = np.max(np.abs(psi - analytic))
check2("2c n=1 excited-state phase test (1D)", err_n1 < 1e-4, f"err={err_n1:.2e}")

for ndim, lengths, ns in [(2, (20.0, 20.0), (96, 96)), (3, (14.0, 14.0, 14.0), (48, 48, 48))]:
    grid_nd = g.make_grid(lengths, ns, boundary='periodic')
    V_nd = pot.harmonic_well(grid_nd, omega)
    psi0_nd, E0_nd = pot.harmonic_eigenstate(grid_nd, omega, n=0)
    n_steps = round(T / dt)
    psi = psi0_nd.copy()
    k2 = prop.kinetic_eigenvalues(grid_nd)
    for _ in range(n_steps):
        psi = prop.strang_step(psi, grid_nd, V_nd, dt, k2=k2)
    analytic = psi0_nd * np.exp(-1j * E0_nd * T)
    err_nd = np.max(np.abs(psi - analytic))
    check2(f"2c ground-state phase test ({ndim}D)", err_nd < 1e-4, f"err={err_nd:.2e}")


PASS: 2c n=1 excited-state phase test (1D)  err=9.48e-06


PASS: 2c ground-state phase test (2D)  err=7.89e-06


PASS: 2c ground-state phase test (3D)  err=8.89e-06


In [11]:
n_pass2 = sum(1 for _, ok, _ in results2 if ok)
print(f"\n{n_pass2}/{len(results2)} Phase 2 checks passed")
assert n_pass2 == len(results2), "Phase 2 validation failed"



6/6 Phase 2 checks passed


## Phase 3 — Observables & absorbing boundary

`observables.py` is checked first against a case with a known closed-form answer
(a free-particle wavepacket's `<x>`/`<p>` are exactly the classical ballistic
trajectory and a conserved momentum), then the complex absorbing potential (CAP,
`potentials.absorbing_boundary`) is checked two ways: that norm stays exactly
conserved with the CAP off (isolates the CAP as the only possible loss source),
and that with the CAP on, a wavepacket sent into it is absorbed essentially
completely with no detectable reflection back into the interior -- the CAP
parameters below (width=15, eta=15, order=3) were tuned by direct simulation
(swept over width/eta/order, checking late-time density in an untouched "quiet"
region of the domain) rather than derived analytically; see `potentials.
absorbing_boundary`'s docstring.

In [12]:
import observables as obs

results3 = []
def check3(name, cond, detail=""):
    results3.append((name, bool(cond), detail))
    print(f"{'PASS' if cond else 'FAIL'}: {name}  {detail}")


In [13]:
# --- Check 3a: free-particle <x>/<p> match the exact classical ballistic
# trajectory (x0+k0*t, constant p=k0) -- validates observables.py's
# expectation-value code, independent of the CAP ---
L, N = 100.0, 512
grid_free = g.make_grid((L,), (N,), boundary='periodic')
sigma0, k0, x0 = 2.0, 1.5, -10.0
psi0 = g.gaussian_wavepacket(grid_free, center=x0, sigma=sigma0, k0=k0)
V0 = np.zeros(grid_free.shape)

dt, T = 0.02, 5.0
n_steps = round(T / dt)
psi = psi0.copy()
k2 = prop.kinetic_eigenvalues(grid_free)
for _ in range(n_steps):
    psi = prop.strang_step(psi, grid_free, V0, dt, k2=k2)

x_mean = obs.expectation_position(psi, grid_free)[0]
p_mean = obs.expectation_momentum(psi, grid_free)[0]
x_expected = x0 + k0 * T
check3("3a <x>(T) matches ballistic trajectory x0+k0*T", abs(x_mean - x_expected) < 1e-6,
       f"<x>={x_mean:.6f} expected={x_expected:.6f}")
check3("3a <p> stays exactly k0 (no potential to change it)", abs(p_mean - k0) < 1e-10,
       f"<p>={p_mean:.10f} expected={k0}")


PASS: 3a <x>(T) matches ballistic trajectory x0+k0*T  <x>=-2.500000 expected=-2.500000
PASS: 3a <p> stays exactly k0 (no potential to change it)  <p>=1.5000000000 expected=1.5


In [14]:
# --- Check 3b: with the CAP off, norm is conserved to machine precision
# even after the packet has wrapped around the periodic domain -- isolates
# the CAP (not the propagator or observables.norm) as the only thing that
# should ever cost norm ---
n0 = obs.norm(psi0, grid_free)
n_final = obs.norm(psi, grid_free)
check3("3b norm conserved with no CAP", abs(n_final - n0) / n0 < 1e-10,
       f"n0={n0:.10f} n_final={n_final:.10f}")


PASS: 3b norm conserved with no CAP  n0=1.0000000000 n_final=1.0000000000


In [15]:
# --- Check 3c: with the CAP on, a wavepacket sent into it is absorbed
# essentially completely, with no detectable reflection left behind in an
# untouched "quiet" region of the interior ---
L, N = 100.0, 1024
grid_cap = g.make_grid((L,), (N,), boundary='periodic')
x = grid_cap.axes[0]
sigma0, k0, x0 = 2.0, 3.0, -30.0
psi0c = g.gaussian_wavepacket(grid_cap, center=x0, sigma=sigma0, k0=k0)
W = pot.absorbing_boundary(grid_cap, width=15.0, eta=15.0, order=3)
V_eff = -1j * W

dt, T = 0.02, 45.0  # long enough that an unabsorbed packet would have
                    # fully crossed the domain by now
n_steps = round(T / dt)
psi = psi0c.copy()
k2 = prop.kinetic_eigenvalues(grid_cap)
for _ in range(n_steps):
    psi = prop.strang_step(psi, grid_cap, V_eff, dt, k2=k2)

remaining_norm = obs.norm(psi, grid_cap)
quiet_density = obs.region_probability(psi, grid_cap, axis=0, x_min=-20, x_max=20)
check3("3c CAP absorbs essentially all incident probability", remaining_norm < 1e-6,
       f"remaining norm={remaining_norm:.2e} (started at 1)")
check3("3c no detectable reflection in the untouched interior", quiet_density < 1e-6,
       f"quiet-zone density={quiet_density:.2e}")


PASS: 3c CAP absorbs essentially all incident probability  remaining norm=1.30e-08 (started at 1)
PASS: 3c no detectable reflection in the untouched interior  quiet-zone density=3.18e-10


In [16]:
n_pass3 = sum(1 for _, ok, _ in results3 if ok)
print(f"\n{n_pass3}/{len(results3)} Phase 3 checks passed")
assert n_pass3 == len(results3), "Phase 3 validation failed"



5/5 Phase 3 checks passed


## Phase 4 — Stationary states (`stationary_states.py`)

A genuinely different numerical method from the spectral propagator: an ordinary
3-point finite-difference Laplacian, Kronecker-summed across axes, diagonalized
with `eigsh`. Its own truncation error is independent of the propagator's, so
agreement with known analytic bound-state energies is a real, independent check
-- not just re-testing Phases 1-3 a different way. Checked against the two
textbook cases with closed-form spectra: the infinite square well
(`E_n=(n*pi/L)^2/2`) and the harmonic well (`E_n=omega*(n+1/2)` per axis, so
`ndim*omega/2` for the isotropic ground state, with the expected degeneracies in
2D/3D), plus a convergence-order check (halving the grid spacing should reduce
the eigenvalue error by ~4x, the signature of a 2nd-order finite-difference
scheme).

In [17]:
import stationary_states as ss

results4 = []
def check4(name, cond, detail=""):
    results4.append((name, bool(cond), detail))
    print(f"{'PASS' if cond else 'FAIL'}: {name}  {detail}")


In [18]:
# --- Check 4a: 1D infinite square well matches E_n=(n*pi/L)^2/2 ---
L, N = 10.0, 400
grid_well = g.make_grid((L,), (N,), boundary='box')
V_well = np.zeros(grid_well.shape)
energies, states = ss.lowest_states(grid_well, V_well, k=5)
analytic = np.array([(n * np.pi / L) ** 2 / 2 for n in range(1, 6)])
rel_err = np.max(np.abs(energies - analytic) / analytic)
check4("4a infinite square well eigenvalues match analytic E_n", rel_err < 5e-4,
       f"max rel err={rel_err:.2e}")


PASS: 4a infinite square well eigenvalues match analytic E_n  max rel err=1.28e-04


In [19]:
# --- Check 4b: 2nd-order convergence -- halving the grid spacing should
# reduce the ground-state eigenvalue error by ~4x ---
ground_analytic = (1 * np.pi / L) ** 2 / 2
errs4 = []
for N_conv in (100, 200, 400):
    grid_conv = g.make_grid((L,), (N_conv,), boundary='box')
    e_conv, _ = ss.lowest_states(grid_conv, np.zeros(grid_conv.shape), k=1)
    errs4.append(abs(e_conv[0] - ground_analytic))
    print(f"  N={N_conv}: ground state err = {errs4[-1]:.3e}")
order4a = np.log2(errs4[0] / errs4[1])
order4b = np.log2(errs4[1] / errs4[2])
check4("4b finite-difference eigenvalues are ~2nd order in grid spacing", 1.5 < order4a < 2.5 and 1.5 < order4b < 2.5,
       f"observed orders: {order4a:.2f}, {order4b:.2f}")


  N=100: ground state err = 3.979e-06
  N=200: ground state err = 1.005e-06
  N=400: ground state err = 2.524e-07
PASS: 4b finite-difference eigenvalues are ~2nd order in grid spacing  observed orders: 1.99, 1.99


In [20]:
# --- Check 4c: 1D harmonic well matches E_n=omega*(n+1/2) ---
L, N, omega = 30.0, 400, 1.0
grid_ho4 = g.make_grid((L,), (N,), boundary='periodic')
V_ho4 = pot.harmonic_well(grid_ho4, omega)
energies_ho, _ = ss.lowest_states(grid_ho4, V_ho4, k=5)
analytic_ho = np.array([omega * (n + 0.5) for n in range(5)])
rel_err_ho = np.max(np.abs(energies_ho - analytic_ho) / analytic_ho)
check4("4c harmonic well eigenvalues match analytic E_n", rel_err_ho < 3e-3,
       f"max rel err={rel_err_ho:.2e}")


PASS: 4c harmonic well eigenvalues match analytic E_n  max rel err=1.60e-03


In [21]:
# --- Check 4d: 2D/3D isotropic harmonic well -- ground state energy and
# the expected degeneracy of the first excited manifold (2-fold in 2D,
# 3-fold in 3D) ---
L, N, omega = 20.0, 80, 1.0
grid2d = g.make_grid((L, L), (N, N), boundary='periodic')
V2d = pot.harmonic_well(grid2d, omega)
e2d, _ = ss.lowest_states(grid2d, V2d, k=4)
check4("4d 2D ground state E0=omega", abs(e2d[0] - omega) / omega < 1e-2, f"E0={e2d[0]:.5f}")
check4("4d 2D first excited pair is degenerate", abs(e2d[1] - e2d[2]) < 1e-3, f"E1={e2d[1]:.5f} E2={e2d[2]:.5f}")
check4("4d 2D first excited energy matches 2*omega", abs(e2d[1] - 2 * omega) / (2 * omega) < 1e-2, f"E1={e2d[1]:.5f}")

L3, N3 = 14.0, 30
grid3d = g.make_grid((L3, L3, L3), (N3, N3, N3), boundary='periodic')
V3d = pot.harmonic_well(grid3d, omega)
e3d, _ = ss.lowest_states(grid3d, V3d, k=5)
check4("4d 3D ground state E0=1.5*omega", abs(e3d[0] - 1.5 * omega) / (1.5 * omega) < 2e-2, f"E0={e3d[0]:.5f}")
check4("4d 3D first excited triplet is degenerate", max(e3d[1:4]) - min(e3d[1:4]) < 1e-3,
       f"E1,E2,E3={e3d[1]:.5f},{e3d[2]:.5f},{e3d[3]:.5f}")
check4("4d 3D first excited energy matches 2.5*omega", abs(e3d[1] - 2.5 * omega) / (2.5 * omega) < 2e-2, f"E1={e3d[1]:.5f}")


PASS: 4d 2D ground state E0=omega  E0=0.99608
PASS: 4d 2D first excited pair is degenerate  E1=1.98820 E2=1.98820
PASS: 4d 2D first excited energy matches 2*omega  E1=1.98820


PASS: 4d 3D ground state E0=1.5*omega  E0=1.47929
PASS: 4d 3D first excited triplet is degenerate  E1,E2,E3=2.45127,2.45127,2.45127
PASS: 4d 3D first excited energy matches 2.5*omega  E1=2.45127


In [22]:
n_pass4 = sum(1 for _, ok, _ in results4 if ok)
print(f"\n{n_pass4}/{len(results4)} Phase 4 checks passed")
assert n_pass4 == len(results4), "Phase 4 validation failed"



9/9 Phase 4 checks passed
